In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
from sklearn.metrics import normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


In [2]:
DATA_PATH = "/kaggle/input/datasets/szishanali/situations-tweets/situations_tweets.csv"  # <-- change if needed
import os
if not os.path.exists(DATA_PATH):
    DATA_PATH = "situations_tweets.csv"

raw = pd.read_csv(DATA_PATH)
KEEP = ["tweet_id", "situation_type", "true_context", "tweet_text",
        "feat_lat", "feat_lon", "timestamp", "date", "day_type", "slot"]
tweets = raw[KEEP].copy()

print(tweets.shape)
tweets.groupby("situation_type").agg(
    annotated_tweets=("tweet_id", "count"),
    distinct_contexts=("true_context", "nunique"),
)


(300, 10)


,annotated_tweets,distinct_contexts
situation_type,,
Earthquake,40,7
Fire,38,5
National Events,22,7
Power Outage,32,6
Protest & Vandalism,26,7
Religious Event,36,8
Telecom Failures,24,7
Terrorism,28,5
Transportation Faults,24,5


In [3]:
EARTH_R_KM = 6371.0
HOUR_KM_PER_HOUR = 4.0
WEEKEND_MISMATCH_KM = 15.0

def build_distance_matrix(df):
    lat = np.radians(df["feat_lat"].values); lon = np.radians(df["feat_lon"].values)
    hour = pd.to_datetime(df["timestamp"]).dt.hour.values.astype(float)
    wknd = (df["day_type"] == "Weekend").astype(int).values
    dlat = lat[:, None] - lat[None, :]; dlon = lon[:, None] - lon[None, :]
    a = np.sin(dlat/2)**2 + np.cos(lat[:,None])*np.cos(lat[None,:])*np.sin(dlon/2)**2
    d_geo = 2 * EARTH_R_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    hd = np.abs(hour[:,None] - hour[None,:]); hd = np.minimum(hd, 24-hd)
    d_time = HOUR_KM_PER_HOUR * hd
    d_day = WEEKEND_MISMATCH_KM * (wknd[:,None] != wknd[None,:]).astype(float)
    D = d_geo + d_time + d_day
    np.fill_diagonal(D, 0.0)
    return D

def hungarian_accuracy(true_labels, pred_labels):
    true_labels = np.asarray(true_labels); pred_labels = np.asarray(pred_labels)
    true_classes = sorted(set(true_labels)); pred_classes = sorted(set(pred_labels) - {-1})
    if not pred_classes:
        return 0, len(true_labels), {}
    contingency = np.zeros((len(pred_classes), len(true_classes)), dtype=int)
    for i, pc in enumerate(pred_classes):
        for j, tc in enumerate(true_classes):
            contingency[i, j] = np.sum((pred_labels == pc) & (true_labels == tc))
    row_ind, col_ind = linear_sum_assignment(-contingency)
    correct = contingency[row_ind, col_ind].sum()
    mapping = {pred_classes[r]: true_classes[c] for r, c in zip(row_ind, col_ind)}
    return int(correct), len(true_labels), mapping

def generic_min_cluster_size(n_tweets, n_true_contexts):
    """One rule, applied identically to every situation type -- not hand-picked."""
    return max(2, round(n_tweets / n_true_contexts / 1.5))


In [4]:
rows, all_dfs = [], []
for situation in sorted(tweets["situation_type"].unique()):
    df = tweets[tweets["situation_type"] == situation].reset_index(drop=True).copy()
    n_tweets = len(df)
    n_true_contexts = df["true_context"].nunique()
    mcs = generic_min_cluster_size(n_tweets, n_true_contexts)

    D = build_distance_matrix(df)
    labels = HDBSCAN(min_cluster_size=mcs, min_samples=2, metric="precomputed").fit_predict(D)
    df["predicted_cluster"] = labels

    correct, total, mapping = hungarian_accuracy(df["true_context"].values, labels)
    nmi = normalized_mutual_info_score(df["true_context"].values, labels)
    df["mapped_true_context"] = df["predicted_cluster"].map(mapping)
    wrong = df[df["mapped_true_context"] != df["true_context"]]
    confused_with = wrong["true_context"].value_counts().index[0] if len(wrong) else "(none - perfect separation)"
    n_clusters_found = len(set(labels) - {-1})
    all_dfs.append(df)

    rows.append({
        "Situation Type": situation, "min_cluster_size_used": mcs,
        "Distinct Contexts (true)": n_true_contexts, "Distinct Contexts (found)": n_clusters_found,
        "Annotated Tweets": n_tweets, "Correctly Clustered": correct,
        "Accuracy (%)": round(100 * correct / total, 1), "NMI Score": round(nmi, 3),
        "Common Misclassified": confused_with,
    })

final_df = pd.DataFrame(rows)
full_tweets = pd.concat(all_dfs, ignore_index=True)
final_df


,Situation Type,min_cluster_size_used,Distinct Contexts (true),Distinct Contexts (found),Annotated Tweets,Correctly Clustered,Accuracy (%),NMI Score,Common Misclassified
0,Earthquake,4,7,7,40,36,90.0,0.851,San Francisco Bay
1,Fire,5,5,4,38,30,78.9,0.900,West London
2,National Events,2,7,7,22,12,54.5,0.611,Washington DC Mall
3,Power Outage,4,6,6,32,27,84.4,0.765,Brooklyn Grid
4,Protest & Vandalism,2,7,7,26,19,73.1,0.726,Barcelona Catalunya
5,Religious Event,3,8,8,36,29,80.6,0.779,Jerusalem Old City
6,Telecom Failures,2,7,7,24,19,79.2,0.748,Shenzhen Grid
7,Terrorism,4,5,5,28,24,85.7,0.781,Central Paris
8,Transportation Faults,3,5,5,24,19,79.2,0.720,Tokyo Rail Loop
9,Weather (Storm),3,6,6,30,24,80.0,0.738,Miami Coast


## 4. Overall totals

In [5]:
total_tweets = final_df["Annotated Tweets"].sum()
total_correct = final_df["Correctly Clustered"].sum()
print(f"TOTAL: {total_correct} / {total_tweets} correctly clustered = {round(100*total_correct/total_tweets, 1)}% overall accuracy")


TOTAL: 239 / 300 correctly clustered = 79.7% overall accuracy


In [6]:
final_df.to_csv("results_table_no_target.csv", index=False)
full_tweets.to_csv("situations_tweets_score.csv", index=False)
print("Saved results_table_no_target.csv and situations_tweets_score.csv")


Saved results_table_no_target.csv and situations_tweets_score.csv
